In [1]:
%pip install -U sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Install required packages
%pip install -U FlagEmbedding pandas openpyxl

# Run the grant ranking script
#!python rank_grants.py "Other Languages.xlsx" \
 #   --profile "We are a women-led NGO in Armenia focused on gender equality and women's rights"

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [9]:
# ============================================================
# ARDOOK DATA GRANT RECOMMENDATION SYSTEM
# WITH SYNTHETIC DATASET TESTING + HIDDEN TEST SET
# ============================================================
# pip install pandas numpy openpyxl sentence-transformers scikit-learn

import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# 1. FILE PATHS, MODEL AND SETTINGS
# ============================================================

FILE_PATH = "Copy of Ardook_Data_All_Grant_Recommendations_UPDATED_v3.xlsx.xlsm"
TEST_FILE = "synthetic_grant_recommendations_format.xlsx"
HIDDEN_TEST_FILE = "Ardook_hidden_test_grants.xlsx"
OUTPUT_FILE = "Ardook_Grant_Recommendation_Results.xlsx"
HIDDEN_TXT_FILE = "Hidden_Test_Results.txt"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Which score is used for classification.
# Options: "Best_Match_Score", "Similarity_Score", "Relevance_Score"
# (Section 9B prints the AUC of each so you can compare them.)
SCORE_COLUMN = "Best_Match_Score"

print("Loading SentenceTransformer model...")
model = SentenceTransformer(MODEL_NAME)

# ============================================================
# 2. SEARCH QUERIES
# ============================================================

search_queries = [
    "Grants for female founders and female entrepreneurs",
    "Funding for women-owned businesses",
    "Grants for women-led startups",
    "Funding for women in technology and AI",
    "Grants for small business owners",
    "Funding for small and independently owned businesses",
    "Grants for female-owned technology companies",
    "Funding for mothers who own businesses",
    "Grants supporting women entrepreneurs and business development"
]

# ============================================================
# 3. KEYWORDS
# ============================================================

keywords = [
    "female",
    "women",
    "woman",
    "women-owned",
    "woman-owned",
    "women-led",
    "female founder",
    "female-owned",
    "small business",
    "small-business",
    "small business owner",
    "entrepreneur",
    "mother"
]

# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

def prepare_grant_text(df):
    """Combine grant information into one text column."""

    possible_columns = [
        "Program",
        "Type",
        "Core Requirements",
        "Why It Fits Ardook Data",
        "Why It Fits",
        "Key Concern / Disqualifier"
    ]

    available_columns = [c for c in possible_columns if c in df.columns]

    # Use only one of the two "Why It Fits" columns
    if ("Why It Fits Ardook Data" in available_columns
            and "Why It Fits" in available_columns):
        available_columns.remove("Why It Fits")

    required_columns = ["Program", "Type", "Core Requirements"]
    missing = [c for c in required_columns if c not in available_columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df["Combined_Text"] = (
        df[available_columns]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
    )
    return df


def calculate_grant_scores(df):
    """Semantic similarity, keyword matches and combined relevance score."""

    grant_embeddings = model.encode(
        df["Combined_Text"].tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    query_embeddings = model.encode(
        search_queries,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    similarity_matrix = cosine_similarity(grant_embeddings, query_embeddings)

    df["Similarity_Score"] = similarity_matrix.mean(axis=1)   # average
    df["Best_Match_Score"] = similarity_matrix.max(axis=1)    # maximum

    def find_matched_keywords(text):
        text = str(text).lower()
        return [k for k in keywords if k.lower() in text]

    df["Matched_Keywords"] = df["Combined_Text"].apply(find_matched_keywords)
    df["Keyword_Count"] = df["Matched_Keywords"].apply(len)

    df["Relevance_Score"] = (
        df["Similarity_Score"] * 80
        + np.minimum(df["Keyword_Count"], 5) * 4
    )
    return df


def categorize_status(row):
    """Categorize grant opportunities using their status and type."""

    status = str(row.get("Current Status (as last reported)", "")).lower()
    grant_type = str(row.get("Type", "")).lower()

    if "closed" in status:
        return "Closed"

    non_grant_types = ["accelerator", "pitch platform", "competition", "challenge"]
    if (any(t in grant_type for t in non_grant_types)
            and "grant" not in grant_type
            and "funding" not in grant_type):
        return "Review - Not Confirmed Grant"

    watchlist_terms = ["watchlist", "monitor", "not verified", "coming soon"]
    if any(t in status for t in watchlist_terms):
        return "Watchlist"

    return "Potentially Applicable"


def normalize_fit(series):
    """Clean the Fit column so it matches fit_mapping."""
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace("–", "-", regex=False)
        .str.replace("—", "-", regex=False)
    )


def safe_auc(y_true, scores):
    """ROC-AUC that returns NaN instead of crashing on a single class."""
    if len(set(y_true)) < 2:
        return float("nan")
    return roc_auc_score(y_true, scores)


# Excellent and Strong = Relevant (1); Moderate / Weak = Not Relevant (0)
fit_mapping = {
    "excellent": 1,
    "strong": 1,
    "moderate": 0,
    "weak": 0,
    "weak-moderate": 0
}

# ============================================================
# 5. LOAD ORIGINAL ARDOOK DATA WORKBOOK
# ============================================================

print("\n========================================")
print("LOADING ARDOOK DATA GRANTS")
print("========================================")

df = pd.read_excel(FILE_PATH, sheet_name="Grant Tracker")
print("Total original grants:", len(df))

df = prepare_grant_text(df)
df = calculate_grant_scores(df)
df["Status_Category"] = df.apply(categorize_status, axis=1)

print("Original grant scoring completed.")

# ============================================================
# 6. LOAD SYNTHETIC TESTING DATASET
# ============================================================

print("\n========================================")
print("LOADING SYNTHETIC TEST DATA")
print("========================================")

synthetic_df = pd.read_excel(TEST_FILE, sheet_name="Grant Tracker")
print("Total synthetic grants:", len(synthetic_df))
print("\nFit rating distribution:")
print(synthetic_df["Fit"].value_counts())

# ============================================================
# 7. CREATE ACTUAL RELEVANCE LABELS
# ============================================================

synthetic_df["Fit_Normalized"] = normalize_fit(synthetic_df["Fit"])
synthetic_df["Actual_Relevant"] = synthetic_df["Fit_Normalized"].map(fit_mapping)

# Exclude ratings without an explicit mapping
synthetic_df = synthetic_df.dropna(subset=["Actual_Relevant"]).copy()
synthetic_df["Actual_Relevant"] = synthetic_df["Actual_Relevant"].astype(int)

print("\nActual relevance labels:")
print(
    synthetic_df["Actual_Relevant"]
    .value_counts()
    .rename({1: "Relevant", 0: "Not Relevant"})
)

# ============================================================
# 8. SCORE SYNTHETIC GRANTS
# ============================================================

synthetic_df = prepare_grant_text(synthetic_df)
synthetic_df = calculate_grant_scores(synthetic_df)

# ============================================================
# 9. SPLIT TRAINING AND TESTING DATA
# ============================================================

y = synthetic_df["Actual_Relevant"]

train_idx, test_idx = train_test_split(
    synthetic_df.index,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train = synthetic_df.loc[train_idx, SCORE_COLUMN]
X_test = synthetic_df.loc[test_idx, SCORE_COLUMN]
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

print("\n========================================")
print("TRAINING AND TESTING SPLIT")
print("========================================")
print("Score column used:", SCORE_COLUMN)
print("Total labeled grants:", len(synthetic_df))
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("\nTraining label distribution:")
print(y_train.value_counts())
print("\nTesting label distribution:")
print(y_test.value_counts())

# ============================================================
# 9B. COMPARE CANDIDATE SCORES BY AUC (TRAINING DATA ONLY)
# ============================================================

score_auc_rows = []
for col in ["Best_Match_Score", "Similarity_Score", "Relevance_Score"]:
    score_auc_rows.append({
        "Score": col,
        "Training_AUC": safe_auc(y_train, synthetic_df.loc[train_idx, col])
    })
score_auc_df = pd.DataFrame(score_auc_rows)

print("\n========================================")
print("TRAINING AUC BY SCORE TYPE (higher = better ranking)")
print("========================================")
print(score_auc_df.to_string(index=False))

# ============================================================
# 10. SELECT BEST THRESHOLD USING TRAINING DATA
# ============================================================
# Grid spans the actual range of the chosen score, so it also works
# for Relevance_Score (which is not limited to 0-1).
# Selection uses balanced accuracy first, then F1, so that
# "predict everything as relevant" is not rewarded.

thresholds = np.linspace(X_train.min(), X_train.max(), 100)

threshold_results = []

for threshold in thresholds:
    train_predictions = (X_train >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_train, train_predictions, zero_division=0),
        "Recall": recall_score(y_train, train_predictions, zero_division=0),
        "F1_Score": f1_score(y_train, train_predictions, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_train, train_predictions)
    })

threshold_df = pd.DataFrame(threshold_results)

best_row = threshold_df.sort_values(
    by=["Balanced_Accuracy", "F1_Score"],
    ascending=False
).iloc[0]

best_threshold = float(best_row["Threshold"])

print("\n========================================")
print("BEST TRAINING THRESHOLD")
print("========================================")
print(f"Best threshold:             {best_threshold:.4f}")
print(f"Training precision:         {best_row['Precision']:.4f}")
print(f"Training recall:            {best_row['Recall']:.4f}")
print(f"Training F1-score:          {best_row['F1_Score']:.4f}")
print(f"Training balanced accuracy: {best_row['Balanced_Accuracy']:.4f}")

if best_threshold <= thresholds[1] or best_threshold >= thresholds[-2]:
    print("\nWARNING: the best threshold is at the edge of the search range.")
    print("The score may not separate relevant from non-relevant grants well.")

# ============================================================
# 11. TEST THE MODEL (SYNTHETIC TEST SPLIT)
# ============================================================

y_pred = (X_test >= best_threshold).astype(int)

accuracy = accuracy_score(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
test_auc = safe_auc(y_test, X_test)

print("\n========================================")
print("FINAL TESTING RESULTS (SYNTHETIC TEST SPLIT)")
print("========================================")
print(f"Threshold:         {best_threshold:.4f}")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Precision:         {precision:.4f}")
print(f"Recall:            {recall:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"ROC-AUC:           {test_auc:.4f}")

# ============================================================
# 12. CLASSIFICATION REPORT
# ============================================================

print("\n========================================")
print("CLASSIFICATION REPORT")
print("========================================")
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["Not Relevant", "Relevant"],
        zero_division=0
    )
)

# ============================================================
# 13. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

print("\n========================================")
print("CONFUSION MATRIX")
print("========================================")
print("Rows = Actual | Columns = Predicted")
print("                 Predicted 0   Predicted 1")
print(f"Actual 0             {cm[0, 0]:<12}{cm[0, 1]}")
print(f"Actual 1             {cm[1, 0]:<12}{cm[1, 1]}")

tn, fp, fn, tp = cm.ravel()
print("\nTrue Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

# ============================================================
# 14. INDIVIDUAL TEST PREDICTIONS
# ============================================================

test_results = synthetic_df.loc[test_idx].copy()
test_results["Predicted_Relevant"] = y_pred.values

test_results["Prediction_Result"] = np.where(
    test_results["Actual_Relevant"] == test_results["Predicted_Relevant"],
    "Correct",
    "Incorrect"
)
test_results["Actual_Label"] = test_results["Actual_Relevant"].map(
    {1: "Relevant", 0: "Not Relevant"}
)
test_results["Predicted_Label"] = test_results["Predicted_Relevant"].map(
    {1: "Relevant", 0: "Not Relevant"}
)

print("\n========================================")
print("INDIVIDUAL TEST PREDICTIONS")
print("========================================")

display_columns = [
    "Program", "Fit", SCORE_COLUMN,
    "Actual_Label", "Predicted_Label", "Prediction_Result"
]
print(
    test_results[display_columns]
    .sort_values(by=SCORE_COLUMN, ascending=False)
    .to_string(index=False)
)

# ============================================================
# 15. THRESHOLD COMPARISON ON TEST DATA
# ============================================================
# Diagnostic only. Do NOT use this table to choose the threshold.

test_threshold_results = []

for threshold in thresholds:
    predictions = (X_test >= threshold).astype(int)
    test_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1_Score": f1_score(y_test, predictions, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_test, predictions)
    })

test_threshold_df = pd.DataFrame(test_threshold_results)

print("\n========================================")
print("TEST THRESHOLD COMPARISON (every 10th threshold)")
print("========================================")
print(test_threshold_df.iloc[::10].to_string(index=False))

# ============================================================
# 16. APPLY SELECTED THRESHOLD TO ORIGINAL GRANTS
# ============================================================

df["Strong_Match"] = df[SCORE_COLUMN] >= best_threshold

df["Recommendation_Status"] = np.where(
    df["Strong_Match"] & (df["Status_Category"] == "Potentially Applicable"),
    "Strong Match",
    "Needs Review"
)

df = df.sort_values(
    by=["Relevance_Score", "Best_Match_Score"],
    ascending=False
).reset_index(drop=True)
df["Rank"] = range(1, len(df) + 1)

best_grants = df[
    df["Strong_Match"] & (df["Status_Category"] == "Potentially Applicable")
].copy()

best_grants = best_grants.sort_values(
    by=["Relevance_Score", "Best_Match_Score"],
    ascending=False
).head(5)
best_grants["Rank"] = range(1, len(best_grants) + 1)

print("\n========================================")
print("TOP 5 ARDOOK DATA GRANT RECOMMENDATIONS")
print("========================================")

top_columns = [
    c for c in [
        "Rank", "Program", "Type", "Relevance_Score", "Best_Match_Score",
        "Status_Category", "Core Requirements",
        "Why It Fits Ardook Data", "Key Concern / Disqualifier"
    ] if c in best_grants.columns
]
print(best_grants[top_columns].to_string(index=False))

# ============================================================
# 17. HIDDEN TEST SET: LOAD, LABEL, SCORE, EVALUATE
# ============================================================

print("\n========================================")
print("HIDDEN TEST SET")
print("========================================")

hidden_df = pd.read_excel(HIDDEN_TEST_FILE, sheet_name="Grant Tracker")
print("Total hidden grants:", len(hidden_df))

# ---- Labels (edit these to your own ratings) ----
# Used only if the hidden file does not already contain a "Fit" column.
# Keys are the Grant ID values in the hidden file.
# NOTE: these are placeholder labels. Replace with your own ratings,
# and set them BEFORE looking at the model's predictions.
hidden_fit_labels = {
    1:  "Moderate",   # Cisco Global Problem Solver Challenge 2027
    2:  "Weak",       # Women's Fund Armenia (Aragatsotn only)
    3:  "Strong",     # Tory Burch Foundation Fellows Program
    4:  "Moderate",   # AWS Activate (Founders tier)
    5:  "Strong",     # Women Founders Network Fast Pitch 2027
    6:  "Weak",       # IOM ELEVATE Business Grants
    7:  "Strong",     # TiE Women Global Pitch Competition 2026
    8:  "Weak",       # D-Prize Global Competition
    9:  "Weak",       # People in Need - Armenian Civil Society
    10: "Excellent",  # Women Who Tech - Tech Startup Grant
}

if "Fit" not in hidden_df.columns:
    hidden_df["Fit"] = hidden_df["Grant ID"].map(hidden_fit_labels)

# ---- Score ----
hidden_df = prepare_grant_text(hidden_df)
hidden_df = calculate_grant_scores(hidden_df)
hidden_df["Status_Category"] = hidden_df.apply(categorize_status, axis=1)

# Use the threshold chosen on TRAINING data (never re-tuned on hidden data)
hidden_df["Predicted_Relevant"] = (
    hidden_df[SCORE_COLUMN] >= best_threshold
).astype(int)

hidden_df["Predicted_Label"] = hidden_df["Predicted_Relevant"].map(
    {1: "Relevant", 0: "Not Relevant"}
)

hidden_df["Recommendation_Status"] = np.where(
    (hidden_df["Predicted_Relevant"] == 1)
    & (hidden_df["Status_Category"] == "Potentially Applicable"),
    "Strong Match",
    "Needs Review"
)

hidden_df["Actual_Relevant"] = normalize_fit(hidden_df["Fit"]).map(fit_mapping)

hidden_df = hidden_df.sort_values(
    by=["Relevance_Score", "Best_Match_Score"],
    ascending=False
).reset_index(drop=True)
hidden_df["Rank"] = range(1, len(hidden_df) + 1)

# ---- Metrics ----
unmapped = hidden_df[hidden_df["Actual_Relevant"].isna()]
if len(unmapped) > 0:
    print("\nRows dropped (Fit value not recognised):")
    print(unmapped[["Program", "Fit"]].to_string(index=False))

labeled = hidden_df.dropna(subset=["Actual_Relevant"]).copy()
labeled["Actual_Relevant"] = labeled["Actual_Relevant"].astype(int)

y_true_h = labeled["Actual_Relevant"]
y_pred_h = labeled["Predicted_Relevant"]

hidden_accuracy = accuracy_score(y_true_h, y_pred_h)
hidden_balanced = balanced_accuracy_score(y_true_h, y_pred_h)
hidden_precision = precision_score(y_true_h, y_pred_h, zero_division=0)
hidden_recall = recall_score(y_true_h, y_pred_h, zero_division=0)
hidden_f1 = f1_score(y_true_h, y_pred_h, zero_division=0)
hidden_auc = safe_auc(y_true_h, labeled[SCORE_COLUMN])

print("\n========================================")
print("HIDDEN TEST SET METRICS")
print("========================================")
print(f"Threshold used:    {best_threshold:.4f}")
print(f"Labeled rows:      {len(labeled)}")
print(f"Accuracy:          {hidden_accuracy:.4f}")
print(f"Balanced accuracy: {hidden_balanced:.4f}")
print(f"Precision:         {hidden_precision:.4f}")
print(f"Recall:            {hidden_recall:.4f}")
print(f"F1-score:          {hidden_f1:.4f}")
print(f"ROC-AUC:           {hidden_auc:.4f}")

print("\nConfusion matrix (rows = actual, columns = predicted)")
print(confusion_matrix(y_true_h, y_pred_h, labels=[0, 1]))

labeled["Result"] = np.where(y_true_h == y_pred_h, "Correct", "Incorrect")
print("\nPer-grant results:")
print(
    labeled[["Program", "Fit", SCORE_COLUMN, "Predicted_Label",
             "Status_Category", "Result"]]
    .to_string(index=False)
)

hidden_metrics_df = pd.DataFrame({
    "Metric": [
        "Selected Threshold", "Accuracy", "Balanced Accuracy",
        "Precision", "Recall", "F1-score", "ROC-AUC", "Rows"
    ],
    "Value": [
        best_threshold, hidden_accuracy, hidden_balanced,
        hidden_precision, hidden_recall, hidden_f1, hidden_auc, len(labeled)
    ]
})

# ============================================================
# 17B. WRITE HIDDEN TEST RESULTS TO A TXT FILE
# ============================================================
 
hidden_cm = confusion_matrix(y_true_h, y_pred_h, labels=[0, 1])
 
with open(HIDDEN_TXT_FILE, "w", encoding="utf-8") as f:
    f.write("=" * 60 + "\n")
    f.write("HIDDEN TEST SET RESULTS\n")
    f.write("=" * 60 + "\n\n")
 
    f.write(f"Hidden file:       {HIDDEN_TEST_FILE}\n")
    f.write(f"Score column:      {SCORE_COLUMN}\n")
    f.write(f"Threshold used:    {best_threshold:.4f}\n")
    f.write(f"Labeled rows:      {len(labeled)}\n\n")
 
    f.write("-" * 60 + "\n")
    f.write("METRICS\n")
    f.write("-" * 60 + "\n")
    f.write(f"Accuracy:          {hidden_accuracy:.4f}\n")
    f.write(f"Balanced accuracy: {hidden_balanced:.4f}\n")
    f.write(f"Precision:         {hidden_precision:.4f}\n")
    f.write(f"Recall:            {hidden_recall:.4f}\n")
    f.write(f"F1-score:          {hidden_f1:.4f}\n")
    f.write(f"ROC-AUC:           {hidden_auc:.4f}\n\n")
 
    f.write("-" * 60 + "\n")
    f.write("CONFUSION MATRIX (rows = actual, columns = predicted)\n")
    f.write("-" * 60 + "\n")
    f.write("                 Predicted 0   Predicted 1\n")
    f.write(f"Actual 0             {hidden_cm[0, 0]:<12}{hidden_cm[0, 1]}\n")
    f.write(f"Actual 1             {hidden_cm[1, 0]:<12}{hidden_cm[1, 1]}\n\n")
 
    f.write("-" * 60 + "\n")
    f.write("PER-GRANT RESULTS\n")
    f.write("-" * 60 + "\n")
    f.write(
        labeled[["Rank", "Program", "Fit", SCORE_COLUMN, "Predicted_Label",
                 "Status_Category", "Recommendation_Status", "Result"]]
        .to_string(index=False)
    )
    f.write("\n")
 
print("Hidden test results written to:", HIDDEN_TXT_FILE)
 
# ============================================================
# 18. EXPORT ALL RESULTS TO EXCEL
# ============================================================

metrics_df = pd.DataFrame({
    "Metric": [
        "Score Column", "Selected Threshold", "Accuracy", "Balanced Accuracy",
        "Precision", "Recall", "F1-score", "ROC-AUC",
        "Training Records", "Testing Records"
    ],
    "Value": [
        SCORE_COLUMN, best_threshold, accuracy, balanced_acc,
        precision, recall, f1, test_auc,
        len(X_train), len(X_test)
    ]
})

confusion_df = pd.DataFrame(
    cm,
    index=["Actual Not Relevant", "Actual Relevant"],
    columns=["Predicted Not Relevant", "Predicted Relevant"]
)

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="All Grant Rankings", index=False)
    best_grants.to_excel(writer, sheet_name="Top 5 Grants", index=False)
    test_results.to_excel(writer, sheet_name="Testing Predictions", index=False)
    metrics_df.to_excel(writer, sheet_name="Evaluation Metrics", index=False)
    confusion_df.to_excel(writer, sheet_name="Confusion Matrix")
    score_auc_df.to_excel(writer, sheet_name="Score AUC Comparison", index=False)
    threshold_df.to_excel(writer, sheet_name="Training Thresholds", index=False)
    test_threshold_df.to_excel(writer, sheet_name="Testing Thresholds", index=False)
    hidden_df.to_excel(writer, sheet_name="Hidden Test Predictions", index=False)
    hidden_metrics_df.to_excel(writer, sheet_name="Hidden Metrics", index=False)

print("\n========================================")
print("EXPORT COMPLETED")
print("========================================")
print("Results saved to:", OUTPUT_FILE)
print("Original grants:", len(df))
print("Synthetic test grants:", len(test_results))
print("Hidden test grants:", len(hidden_df))
print("Top recommendations:", len(best_grants))

Loading SentenceTransformer model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


LOADING ARDOOK DATA GRANTS
Total original grants: 37
Original grant scoring completed.

LOADING SYNTHETIC TEST DATA
Total synthetic grants: 70

Fit rating distribution:
Fit
Strong       35
Excellent    18
Moderate     17
Name: count, dtype: int64

Actual relevance labels:
Actual_Relevant
Relevant        53
Not Relevant    17
Name: count, dtype: int64

TRAINING AND TESTING SPLIT
Score column used: Best_Match_Score
Total labeled grants: 70
Training records: 56
Testing records: 14

Training label distribution:
Actual_Relevant
1    42
0    14
Name: count, dtype: int64

Testing label distribution:
Actual_Relevant
1    11
0     3
Name: count, dtype: int64

TRAINING AUC BY SCORE TYPE (higher = better ranking)
           Score  Training_AUC
Best_Match_Score      0.551020
Similarity_Score      0.569728
 Relevance_Score      0.583333

BEST TRAINING THRESHOLD
Best threshold:             0.5558
Training precision:         0.9375
Training recall:            0.3571
Training F1-score:          0.517

In [9]:
import pandas as pd
from FlagEmbedding import BGEM3FlagModel
from IPython.display import display

In [6]:
# Input spreadsheet
input_file = "Other_Languages.xlsx"

# Output spreadsheet
output_file = "ranked_grants.xlsx"

# Organization profile
profile = """
We are a women-led NGO in Armenia focused on gender equality,
women's rights, women's empowerment, and supporting women and
girls through social, educational, and community-based initiatives.
"""

# BGE-M3 hybrid scoring weights
WEIGHTS = [0.4, 0.2, 0.4]

# Settings
USE_AMOUNT = False
TOP_N = 0  # 0 = include all grants

In [7]:
# Load Excel spreadsheet
df = pd.read_excel(input_file)

# Check required columns
required_columns = ["Name", "Requirements", "Amount", "Link"]

missing_columns = set(required_columns) - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# Remove rows without grant names or requirements
df = df.dropna(
    subset=["Name", "Requirements"],
    how="all"
).reset_index(drop=True)

print(f"Total grants loaded: {len(df)}")

display(df.head())

Total grants loaded: 3


,Name,Requirements,Amount,Link
0,Կանչ կանանց նորաստեղծ ՔՀԿ-ների համար. Ինստիտու...,"Լինեն կանանց կողմից ղեկավարվող ՔՀԿ, գրանցված Հ...",1 748 000 ՀՀ դրամ,https://womenofarmenia.org/2026/02/23/call-for...
1,Կանանց հիմնադրամ Հայաստանում,Տարբեր հարցերի շուրջ 40 անց կանանց ձայնը լսելի...,1 միլիոն 200 հազար ՀՀ,https://womenfundarmenia.org/%D5%A4%D6%80%D5%A...
2,For Her Future,ინიციატივა დაეხმარება საწარმოებს დანერგონ ინოვ...,"40, 000 ლარი",https://forher.ge/granti/mtsire-da-sashualo-sa...


In [8]:
def build_doc_text(row, use_amount=False):

    parts = [
        str(row["Name"]).strip(),
        str(row["Requirements"]).strip()
    ]

    if use_amount:
        parts.append(str(row["Amount"]).strip())

    # Remove empty or missing values
    parts = [
        p for p in parts
        if p and p.lower() != "nan"
    ]

    return "\n".join(parts)


# Create grant descriptions
docs = [
    build_doc_text(row, USE_AMOUNT)
    for _, row in df.iterrows()
]

print("Grant descriptions prepared:", len(docs))

# Preview one grant
print(docs[0])

Grant descriptions prepared: 3
Կանչ կանանց նորաստեղծ ՔՀԿ-ների համար. Ինստիտուցիոնալ հզորացման դրամաշնորհներ (3-6 ամիս)
Լինեն կանանց կողմից ղեկավարվող ՔՀԿ, գրանցված Հայաստանում վերջին 1–3 տարիների ընթացքում
Գործեն գենդերային հավասարության, կանանց իրավունքների կամ ինտերսեկցիոնալ արդարության ոլորտում
Ցուցաբերեն հանձնառություն մարդու իրավունքների և հաշվետվողականության սկզբունքներին


In [9]:
model = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=True
)

print("BGE-M3 model loaded successfully!")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BGE-M3 model loaded successfully!


In [10]:
# Create profile-grant pairs
pairs = [
    [profile, grant]
    for grant in docs
]

# Calculate similarity scores
scores = model.compute_score(
    pairs,
    max_passage_length=2048,
    weights_of_each_score=WEIGHTS
)

print("Similarity scoring completed!")

Similarity scoring completed!


In [11]:
# Add individual similarity scores
df["score_dense"] = scores["dense"]
df["score_sparse"] = scores["sparse"]
df["score_colbert"] = scores["colbert"]

# Add combined hybrid score
df["score"] = scores["colbert+sparse+dense"]

# Sort grants from highest to lowest similarity
ranked = df.sort_values(
    by="score",
    ascending=False
).reset_index(drop=True)

# Add ranking column
ranked.insert(0, "Rank", ranked.index + 1)

# Optionally display only the top N grants
if TOP_N > 0:
    ranked = ranked.head(TOP_N)

display(ranked.head(20))

,Rank,Name,Requirements,Amount,Link,score_dense,score_sparse,score_colbert,score
0,1,Կանանց հիմնադրամ Հայաստանում,Տարբեր հարցերի շուրջ 40 անց կանանց ձայնը լսելի...,1 միլիոն 200 հազար ՀՀ,https://womenfundarmenia.org/%D5%A4%D6%80%D5%A...,0.540527,0.000000,0.578470,0.372999
1,2,Կանչ կանանց նորաստեղծ ՔՀԿ-ների համար. Ինստիտու...,"Լինեն կանանց կողմից ղեկավարվող ՔՀԿ, գրանցված Հ...",1 748 000 ՀՀ դրամ,https://womenofarmenia.org/2026/02/23/call-for...,0.535645,0.000000,0.501764,0.345803
2,3,For Her Future,ინიციატივა დაეხმარება საწარმოებს დანერგონ ინოვ...,"40, 000 ლარი",https://forher.ge/granti/mtsire-da-sashualo-sa...,0.503906,0.010773,0.505671,0.340117


In [12]:
# Export ranked grants
ranked.to_excel(
    output_file,
    index=False
)

print(f"Ranked grants successfully saved to: {output_file}")

Ranked grants successfully saved to: ranked_grants.xlsx


In [13]:
%pip install deep-translator

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [16]:
%pip install -U transformers sentencepiece langdetect

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [17]:
import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langdetect import detect

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)

translation_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

print("Translation model loaded successfully!")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Translation model loaded successfully!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

In [21]:
import re
import torch
import pandas as pd

# NLLB language codes
LANGUAGE_CODES = {
    "Armenian": "hye_Armn",
    "Georgian": "kat_Geor",
    "English": "eng_Latn",
    "Russian": "rus_Cyrl",
    "French": "fra_Latn",
    "Spanish": "spa_Latn",
    "German": "deu_Latn"
}


# Detect Armenian and Georgian using their Unicode characters
def detect_source_language(text):

    armenian_count = sum(
        0x0530 <= ord(char) <= 0x058F
        for char in text
    )

    georgian_count = sum(
        0x10A0 <= ord(char) <= 0x10FF
        or 0x2D00 <= ord(char) <= 0x2D2F
        for char in text
    )

    if armenian_count > georgian_count and armenian_count > 0:
        return "Armenian"

    if georgian_count > armenian_count and georgian_count > 0:
        return "Georgian"

    # Use automatic detection for other languages
    from langdetect import detect

    detected = detect(text)

    language_map = {
        "en": "English",
        "ru": "Russian",
        "fr": "French",
        "es": "Spanish",
        "de": "German"
    }

    return language_map.get(detected, "Unknown")


# Translate text into English
def translate_to_english(text):

    if pd.isna(text) or str(text).strip() == "":
        return text

    text = str(text).strip()

    try:
        source_language = detect_source_language(text)

        # No translation needed
        if source_language == "English":
            return text

        if source_language == "Unknown":
            print(f"Unsupported language: {text[:50]}")
            return text

        source_code = LANGUAGE_CODES[source_language]

        tokenizer.src_lang = source_code

        # Split long text into manageable token chunks
        token_ids = tokenizer(
            text,
            add_special_tokens=False
        )["input_ids"]

        chunks = [
            token_ids[i:i + 400]
            for i in range(0, len(token_ids), 400)
        ]

        translations = []

        for chunk in chunks:

            chunk_text = tokenizer.decode(
                chunk,
                skip_special_tokens=True
            )

            inputs = tokenizer(
                chunk_text,
                return_tensors="pt",
                truncation=True,
                max_length=512
            )

            device = next(translation_model.parameters()).device
            inputs = {
                key: value.to(device)
                for key, value in inputs.items()
            }

            with torch.no_grad():

                translated_tokens = translation_model.generate(
                    **inputs,
                    forced_bos_token_id=tokenizer.convert_tokens_to_ids(
                        "eng_Latn"
                    ),
                    max_length=512,
                    num_beams=4
                )

            translated_text = tokenizer.batch_decode(
                translated_tokens,
                skip_special_tokens=True
            )[0]

            translations.append(translated_text)

        return " ".join(translations)

    except Exception as e:
        print(f"Translation failed: {e}")
        return text

In [26]:
from tqdm.notebook import tqdm

# Only translate the top 10 grants
ranked_english = ranked.head(10).copy()

# Cache translations to avoid translating duplicate text
translation_cache = {}

def translate_cached(text):

    if pd.isna(text):
        return text

    text = str(text).strip()

    if not text:
        return text

    if text not in translation_cache:
        translation_cache[text] = translate_to_english(text)

    return translation_cache[text]


# Translate grant names
ranked_english["Name_English"] = [
    translate_cached(text)
    for text in tqdm(
        ranked_english["Name"],
        desc="Translating grant names"
    )
]

# Translate grant requirements
ranked_english["Requirements_English"] = [
    translate_cached(text)
    for text in tqdm(
        ranked_english["Requirements"],
        desc="Translating requirements"
    )
]

print("Translation completed!")

Translating grant names:   0%|          | 0/3 [00:00<?, ?it/s]

Translating requirements:   0%|          | 0/3 [00:00<?, ?it/s]

Translation completed!


In [27]:
display_columns = [
    "Rank",
    "Name_English",
    "Requirements_English",
    "Amount",
    "score_dense",
    "score_sparse",
    "score_colbert",
    "score",
    "Link"
]

display(ranked_english[display_columns])

,Rank,Name_English,Requirements_English,Amount,score_dense,score_sparse,score_colbert,score,Link
0,1,The Women's Foundation in Armenia,Activities for women over 40 about women's sex...,1 միլիոն 200 հազար ՀՀ,0.540527,0.000000,0.578470,0.372999,https://womenfundarmenia.org/%D5%A4%D6%80%D5%A...
1,2,Call for women's CSOs: Institutional empowerme...,Women-led NGOs registered in Armenia for the p...,1 748 000 ՀՀ դրամ,0.535645,0.000000,0.501764,0.345803,https://womenofarmenia.org/2026/02/23/call-for...
2,3,For Her Future,The initiative will help enterprises implement...,"40, 000 ლარი",0.503906,0.010773,0.505671,0.340117,https://forher.ge/granti/mtsire-da-sashualo-sa...


In [28]:
ranked_english.to_excel(
    "top_10_ranked_grants_english.xlsx",
    index=False
)

print("Export completed!")

Export completed!


In [11]:
#NEW CODE

# ============================================================
# MULTILINGUAL GRANT RECOMMENDATION SYSTEM
# BGE-M3 HYBRID MODEL + 60/40 EVALUATION
# ============================================================

# Install dependencies:
# pip install -U FlagEmbedding pandas numpy openpyxl scikit-learn

import pandas as pd
import numpy as np

from FlagEmbedding import BGEM3FlagModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# 1. SETTINGS
# ============================================================

# Input spreadsheet
input_file = "Other_Languages.xlsx"

# Output spreadsheet
output_file = "ranked_grants.xlsx"

# Organization profile
profile = """
We are a women-led NGO in Armenia focused on gender equality,
women's rights, women's empowerment, and supporting women and
girls through social, educational, and community-based initiatives.
"""

# BGE-M3 hybrid scoring weights
WEIGHTS = [0.4, 0.2, 0.4]

# Settings
USE_AMOUNT = False
TOP_N = 0  # 0 = include all grants

# Training and testing settings
TRAIN_SIZE = 0.60
TEST_SIZE = 0.40
RANDOM_STATE = 42

# BGE-M3 model
MODEL_NAME = "BAAI/bge-m3"

# ============================================================
# 2. LOAD BGE-M3 MODEL
# ============================================================

print("\nLoading BGE-M3 model...")

model = BGEM3FlagModel(
    MODEL_NAME,
    use_fp16=True
)

print("Model loaded successfully.")

# ============================================================
# 3. LOAD DATASET
# ============================================================

print("\nLoading grant dataset...")

df = pd.read_excel(input_file)

print("Total grants:", len(df))
print("Columns:", df.columns.tolist())

required_columns = [
    "Name",
    "Requirements",
    "Amount"
]

for column in required_columns:
    if column not in df.columns:
        raise ValueError(
            f"Missing required column: {column}"
        )

# Preserve manually entered labels if available
if "Actual_Relevant" in df.columns:
    df["Actual_Relevant"] = pd.to_numeric(
        df["Actual_Relevant"],
        errors="coerce"
    )

# Clean missing values
for column in required_columns:
    df[column] = df[column].fillna("").astype(str)

# ============================================================
# 4. PREPARE GRANT TEXT
# ============================================================

if USE_AMOUNT:
    df["Grant_Text"] = (
        "Grant Name: " + df["Name"]
        + "\nRequirements: " + df["Requirements"]
        + "\nAmount: " + df["Amount"]
    )
else:
    df["Grant_Text"] = (
        "Grant Name: " + df["Name"]
        + "\nRequirements: " + df["Requirements"]
    )

# ============================================================
# 5. CALCULATE BGE-M3 HYBRID SCORES
# ============================================================

print("\nCalculating BGE-M3 scores...")

sentence_pairs = [
    (profile, grant_text)
    for grant_text in df["Grant_Text"].tolist()
]

scores = model.compute_score(
    sentence_pairs,
    batch_size=12,
    max_passage_length=8192,
    weights_for_different_modes=WEIGHTS
)

# Individual scoring components
df["score_dense"] = scores["dense"]
df["score_sparse"] = scores["sparse"]
df["score_colbert"] = scores["colbert"]

# Combined hybrid score
hybrid_key = "colbert+sparse+dense"

if hybrid_key not in scores:
    raise KeyError(
        "Combined BGE-M3 score was not returned. "
        f"Available score keys: {list(scores.keys())}"
    )

df["score"] = scores[hybrid_key]

# ============================================================
# 6. RANK GRANTS
# ============================================================

df = df.sort_values(
    by="score",
    ascending=False
).reset_index(drop=True)

df["Rank"] = range(1, len(df) + 1)

if TOP_N > 0:
    ranked_df = df.head(TOP_N).copy()
else:
    ranked_df = df.copy()

# ============================================================
# 7. PRINT GRANT RANKINGS
# ============================================================

print("\n============================================")
print("MULTILINGUAL GRANT RECOMMENDATIONS")
print("============================================")

print("\nOrganization profile:")
print(profile)

print("\nScoring weights:")
print("Dense:", WEIGHTS[0])
print("Sparse:", WEIGHTS[1])
print("ColBERT:", WEIGHTS[2])

print("\nTotal grants ranked:", len(ranked_df))

print("\n============================================")
print("RANKED GRANTS")
print("============================================")

print(
    ranked_df[
        [
            "Rank",
            "Name",
            "Amount",
            "score_dense",
            "score_sparse",
            "score_colbert",
            "score"
        ]
    ].to_string(index=False)
)

# ============================================================
# 8. CHECK FOR MANUALLY REVIEWED LABELS
# ============================================================

print("\n============================================")
print("EVALUATION METRICS")
print("============================================")

if "Actual_Relevant" not in ranked_df.columns:

    ranked_df["Actual_Relevant"] = np.nan

    print("\nNo Actual_Relevant column found.")

    print("\nPlease manually label the grants:")
    print("1 = Relevant to the NGO profile")
    print("0 = Not relevant to the NGO profile")

    print("\nThe evaluation metrics will be calculated")
    print("after the labels have been added.")

    metrics_available = False

else:

    # Keep only valid 0/1 labels
    labeled_df = ranked_df[
        ranked_df["Actual_Relevant"].isin([0, 1])
    ].copy()

    labeled_df["Actual_Relevant"] = (
        labeled_df["Actual_Relevant"].astype(int)
    )

    print(
        "\nManually labeled grants:",
        len(labeled_df)
    )

    print("\nLabel distribution:")
    print(
        labeled_df["Actual_Relevant"]
        .value_counts()
        .rename({
            1: "Relevant",
            0: "Not Relevant"
        })
    )

    metrics_available = False

    # ========================================================
    # 9. TRAINING AND TESTING SPLIT (60% / 40%)
    # ========================================================

    class_counts = labeled_df[
        "Actual_Relevant"
    ].value_counts()

    if len(class_counts) < 2 or class_counts.min() < 2:

        print(
            "\nNot enough examples in both classes "
            "for a stratified train/test split."
        )

    else:

        X = labeled_df["score"]
        y = labeled_df["Actual_Relevant"]

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            train_size=TRAIN_SIZE,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=y
        )

        print("\n========== DATA SPLIT ==========")

        print("Total labeled grants:", len(labeled_df))
        print("Training records:", len(X_train))
        print("Testing records:", len(X_test))

        print("\nTraining distribution:")
        print(
            y_train.value_counts().rename({
                1: "Relevant",
                0: "Not Relevant"
            })
        )

        print("\nTesting distribution:")
        print(
            y_test.value_counts().rename({
                1: "Relevant",
                0: "Not Relevant"
            })
        )

        # ====================================================
        # 10. SELECT THRESHOLD USING TRAINING DATA
        # ====================================================

        thresholds = np.linspace(
            X_train.min(),
            X_train.max(),
            101
        )

        threshold_results = []

        for threshold in thresholds:

            train_predictions = (
                X_train >= threshold
            ).astype(int)

            train_precision = precision_score(
                y_train,
                train_predictions,
                zero_division=0
            )

            train_recall = recall_score(
                y_train,
                train_predictions,
                zero_division=0
            )

            train_f1 = f1_score(
                y_train,
                train_predictions,
                zero_division=0
            )

            threshold_results.append({
                "Threshold": threshold,
                "Precision": train_precision,
                "Recall": train_recall,
                "F1_Score": train_f1
            })

        threshold_df = pd.DataFrame(
            threshold_results
        )

        # Select the threshold with the highest training F1
        best_row = threshold_df.sort_values(
            by=["F1_Score", "Precision"],
            ascending=False
        ).iloc[0]

        best_threshold = best_row["Threshold"]

        print("\n========== BEST THRESHOLD ==========")

        print(f"Threshold: {best_threshold:.4f}")
        print(f"Training Precision: {best_row['Precision']:.4f}")
        print(f"Training Recall: {best_row['Recall']:.4f}")
        print(f"Training F1-score: {best_row['F1_Score']:.4f}")

        # ====================================================
        # 11. TEST MODEL ON UNSEEN DATA
        # ====================================================

        y_pred = (
            X_test >= best_threshold
        ).astype(int)

        accuracy = accuracy_score(
            y_test,
            y_pred
        )

        precision = precision_score(
            y_test,
            y_pred,
            zero_division=0
        )

        recall = recall_score(
            y_test,
            y_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_test,
            y_pred,
            zero_division=0
        )

        print("\n============================================")
        print("FINAL TESTING RESULTS")
        print("============================================")

        print(f"Threshold: {best_threshold:.4f}")
        print(f"Accuracy:  {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1-score:  {f1:.4f}")

        # ====================================================
        # 12. CLASSIFICATION REPORT
        # ====================================================

        print("\n========== CLASSIFICATION REPORT ==========")

        print(
            classification_report(
                y_test,
                y_pred,
                labels=[0, 1],
                target_names=[
                    "Not Relevant",
                    "Relevant"
                ],
                zero_division=0
            )
        )

        # ====================================================
        # 13. CONFUSION MATRIX
        # ====================================================

        cm = confusion_matrix(
            y_test,
            y_pred,
            labels=[0, 1]
        )

        tn, fp, fn, tp = cm.ravel()

        print("\n========== CONFUSION MATRIX ==========")

        print("Rows = Actual | Columns = Predicted")
        print("                 Predicted 0   Predicted 1")
        print(f"Actual 0             {tn:<12}{fp}")
        print(f"Actual 1             {fn:<12}{tp}")

        print("\nTrue Negatives:", tn)
        print("False Positives:", fp)
        print("False Negatives:", fn)
        print("True Positives:", tp)

        # ====================================================
        # 14. INDIVIDUAL TEST PREDICTIONS
        # ====================================================

        test_results = labeled_df.loc[
            X_test.index
        ].copy()

        test_results["Predicted_Relevant"] = y_pred

        test_results["Actual_Label"] = (
            test_results["Actual_Relevant"].map({
                1: "Relevant",
                0: "Not Relevant"
            })
        )

        test_results["Predicted_Label"] = (
            test_results["Predicted_Relevant"].map({
                1: "Relevant",
                0: "Not Relevant"
            })
        )

        test_results["Prediction_Result"] = np.where(
            test_results["Actual_Relevant"]
            == test_results["Predicted_Relevant"],
            "Correct",
            "Incorrect"
        )

        print("\n========== INDIVIDUAL TEST PREDICTIONS ==========")

        print(
            test_results[
                [
                    "Name",
                    "score",
                    "Actual_Label",
                    "Predicted_Label",
                    "Prediction_Result"
                ]
            ].to_string(index=False)
        )

        # ====================================================
        # 15. EVALUATION OUTPUT TABLES
        # ====================================================

        metrics_df = pd.DataFrame({
            "Metric": [
                "Training Percentage",
                "Testing Percentage",
                "Selected Threshold",
                "Accuracy",
                "Precision",
                "Recall",
                "F1-score",
                "Training Records",
                "Testing Records"
            ],
            "Value": [
                TRAIN_SIZE,
                TEST_SIZE,
                best_threshold,
                accuracy,
                precision,
                recall,
                f1,
                len(X_train),
                len(X_test)
            ]
        })

        confusion_df = pd.DataFrame(
            cm,
            index=[
                "Actual Not Relevant",
                "Actual Relevant"
            ],
            columns=[
                "Predicted Not Relevant",
                "Predicted Relevant"
            ]
        )

        metrics_available = True

# ============================================================
# 16. EXPORT RESULTS TO EXCEL
# ============================================================

print("\nExporting results...")

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    ranked_df.to_excel(
        writer,
        sheet_name="Ranked Grants",
        index=False
    )

    if metrics_available:

        metrics_df.to_excel(
            writer,
            sheet_name="Evaluation Metrics",
            index=False
        )

        confusion_df.to_excel(
            writer,
            sheet_name="Confusion Matrix"
        )

        test_results.to_excel(
            writer,
            sheet_name="Test Predictions",
            index=False
        )

        threshold_df.to_excel(
            writer,
            sheet_name="Training Thresholds",
            index=False
        )

print("\n============================================")
print("PROCESS COMPLETED")
print("============================================")

print("Output file:", output_file)
print("Total grants ranked:", len(ranked_df))

if metrics_available:
    print("Training records:", len(X_train))
    print("Testing records:", len(X_test))
    print("Evaluation metrics exported successfully.")
else:
    print("Grant rankings exported.")
    print("Add manual labels and run again for evaluation.")



Loading BGE-M3 model...


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

... (2 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded successfully.

Loading grant dataset...
Total grants: 3
Columns: ['Name', 'Requirements', 'Amount', 'Link']

Calculating BGE-M3 scores...

MULTILINGUAL GRANT RECOMMENDATIONS

Organization profile:

We are a women-led NGO in Armenia focused on gender equality,
women's rights, women's empowerment, and supporting women and
girls through social, educational, and community-based initiatives.


Scoring weights:
Dense: 0.4
Sparse: 0.2
ColBERT: 0.4

Total grants ranked: 3

RANKED GRANTS
 Rank                                                                                    Name                Amount  score_dense  score_sparse  score_colbert    score
    1                                                            Կանանց հիմնադրամ Հայաստանում 1 միլիոն 200 հազար ՀՀ     0.565430           0.0       0.593521 0.463580
    2                                                                          For Her Future          40, 000 ლარი     0.526855           0.0       0.521191 0.419219
  

In [18]:
import pandas as pd
import re

input_path = "SyntheticDataGeneratedByAI.xlsx"
output_path = "SyntheticDataGeneratedByAI_with_reference_labels.xlsx"

df = pd.read_excel(input_path, sheet_name="Sheet1")

# Multilingual phrases that explicitly indicate a women/gender-equality focus.
gender_terms = [
    # Georgian
    "ქალთა", "ქალების", "ქალთა უფლებ", "გენდერულ თანასწორ",
    "ქალების მიერ მართულ", "ქალთა ეკონომიკურ", "ქალთა მეწარმ",
    "ქალთა გაძლიერ", "ქალებისა და გოგონ",
    # Armenian
    "կանանց", "գենդերային հավասար", "կանանց իրավունք",
    "կանանց կողմից ղեկավարվ", "կանանց հիմնադրած",
    "կանանց տնտեսական", "կանանց և աղջիկ",
    # Russian
    "женщин", "гендерного равенства", "права женщин",
    "под руководством женщин",
    # Arabic
    "النساء", "المرأة", "المساواة بين الجنسين",
    "حقوق المرأة", "تقودها النساء",
    # French
    "femmes", "égalité de genre", "droits des femmes",
    "dirigées par des femmes",
    # Spanish
    "mujeres", "igualdad de género", "derechos de las mujeres",
    "lideradas por mujeres",
]

def contains_gender_focus(text):
    text = str(text).casefold()
    return any(term.casefold() in text for term in gender_terms)

labels = []
rationales = []
statuses = []

for _, row in df.iterrows():
    name = str(row.get("Name", ""))
    req = str(row.get("Requirements", ""))
    name_focus = contains_gender_focus(name)
    req_focus = contains_gender_focus(req)

    if req_focus and name_focus:
        label = 1
        rationale = "Both the grant title and requirements explicitly indicate a women/gender-equality focus."
        status = "Provisional — verify"
    elif req_focus:
        label = 1
        rationale = "The requirements explicitly prioritize women, girls, women's rights, or gender equality."
        status = "Provisional — verify"
    elif name_focus:
        label = 1
        rationale = "The title indicates a women-focused grant, but the requirements do not clearly confirm this; check the source details."
        status = "Needs human review — title/requirements mismatch"
    else:
        label = 0
        rationale = "Neither the title nor the requirements explicitly identify a women/gender-equality focus."
        status = "Provisional — verify"

    labels.append(label)
    rationales.append(rationale)
    statuses.append(status)

df["Actual_Relevant"] = labels
df["Label_Rationale"] = rationales
df["Label_Review_Status"] = statuses

# Keep original source sheet and add a summary sheet.
summary = (
    df.groupby(["Actual_Relevant", "Label_Review_Status"])
      .size()
      .reset_index(name="Count")
)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Labeled_Grants", index=False)
    summary.to_excel(writer, sheet_name="Label_Summary", index=False)

print(f"Created: {output_path}")
print(f"Total grants: {len(df)}")
print(df["Actual_Relevant"].value_counts().sort_index().rename(index={0:"Not relevant (0)", 1:"Relevant (1)"}))
print("\nReview-status counts:")
print(df["Label_Review_Status"].value_counts())


Created: SyntheticDataGeneratedByAI_with_reference_labels.xlsx
Total grants: 100
Actual_Relevant
Not relevant (0)    45
Relevant (1)        55
Name: count, dtype: int64

Review-status counts:
Label_Review_Status
Provisional — verify                                72
Needs human review — title/requirements mismatch    28
Name: count, dtype: int64


In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split

input_path = "SyntheticDataGeneratedByAI_with_reference_labels.xlsx"
output_path = "Grant_Dataset_60_40_Split.xlsx"

df = pd.read_excel(input_path, sheet_name="Labeled_Grants")

# Validate labels before stratified splitting
if "Actual_Relevant" not in df.columns:
    raise ValueError("The Actual_Relevant column is missing.")
if df["Actual_Relevant"].isna().any():
    raise ValueError("Some grants have missing reference labels.")
if df["Actual_Relevant"].nunique() < 2:
    raise ValueError("Stratified splitting requires both label classes.")

train_df, test_df = train_test_split(
    df,
    train_size=0.60,
    test_size=0.40,
    random_state=42,
    shuffle=True,
    stratify=df["Actual_Relevant"]
)

train_df = train_df.copy()
test_df = test_df.copy()

train_df.insert(0, "Dataset_Split", "Training")
test_df.insert(0, "Dataset_Split", "Testing")

combined_df = pd.concat([train_df, test_df], ignore_index=True)

split_summary = pd.DataFrame({
    "Dataset": ["Training", "Testing", "Total"],
    "Records": [len(train_df), len(test_df), len(df)],
    "Relevant (1)": [
        int((train_df["Actual_Relevant"] == 1).sum()),
        int((test_df["Actual_Relevant"] == 1).sum()),
        int((df["Actual_Relevant"] == 1).sum())
    ],
    "Not Relevant (0)": [
        int((train_df["Actual_Relevant"] == 0).sum()),
        int((test_df["Actual_Relevant"] == 0).sum()),
        int((df["Actual_Relevant"] == 0).sum())
    ]
})

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    train_df.to_excel(writer, sheet_name="Training_60", index=False)
    test_df.to_excel(writer, sheet_name="Testing_40", index=False)
    combined_df.to_excel(writer, sheet_name="Combined_100", index=False)
    split_summary.to_excel(writer, sheet_name="Split_Summary", index=False)

print(f"Created: {output_path}")
print(split_summary.to_string(index=False))


Created: Grant_Dataset_60_40_Split.xlsx
 Dataset  Records  Relevant (1)  Not Relevant (0)
Training       60            33                27
 Testing       40            22                18
   Total      100            55                45


In [23]:
import pandas as pd
import numpy as np

from FlagEmbedding import BGEM3FlagModel
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# Input: workbook containing the already-created 60/40 split
input_file = "Grant_Dataset_60_40_Split.xlsx"
output_file = "Grant_BGE_M3_Evaluation_Results.xlsx"

profile = """
We are a women-led NGO in Armenia focused on gender equality,
women's rights, women's empowerment, and supporting women and
girls through social, educational, and community-based initiatives.
"""

WEIGHTS = [0.4, 0.2, 0.4]
BATCH_SIZE = 12

# 1. Load the fixed training and testing sets
train_df = pd.read_excel(input_file, sheet_name="Training_60")
test_df = pd.read_excel(input_file, sheet_name="Testing_40")

for name, data in [("Training", train_df), ("Testing", test_df)]:
    if "Actual_Relevant" not in data.columns:
        raise ValueError(f"{name} sheet is missing Actual_Relevant.")
    if data["Actual_Relevant"].isna().any():
        raise ValueError(f"{name} sheet contains missing labels.")
    if not set(data["Actual_Relevant"].unique()).issubset({0, 1}):
        raise ValueError(f"{name} labels must be 0 or 1.")

# 2. Construct text from all three original input columns
def make_grant_text(data):
    return (
        "Grant Name: " + data["Name"].fillna("").astype(str)
        + "\nRequirements: " + data["Requirements"].fillna("").astype(str)
        + "\nAmount: " + data["Amount"].fillna("").astype(str)
    )

train_df = train_df.copy()
test_df = test_df.copy()
train_df["Grant_Text"] = make_grant_text(train_df)
test_df["Grant_Text"] = make_grant_text(test_df)

# 3. Load pretrained BGE-M3
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)

# 4. Score each set using the same profile and hybrid weights
def score_grants(data):
    pairs = [[profile, text] for text in data["Grant_Text"].tolist()]
    raw = model.compute_score(
        pairs,
        batch_size=BATCH_SIZE,
        max_passage_length=8192,
        weights_for_different_modes=WEIGHTS
    )

    result = data.copy()
    result["score_dense"] = raw["dense"]
    result["score_sparse"] = raw["sparse"]
    result["score_colbert"] = raw["colbert"]

    # FlagEmbedding versions may expose the hybrid key in different orders.
    hybrid_key = next(
        (key for key in raw.keys()
         if isinstance(key, str)
         and set(key.split("+")) == {"dense", "sparse", "colbert"}),
        None
    )
    if hybrid_key is None:
        raise KeyError(
            f"Could not find combined hybrid score in keys: {list(raw.keys())}"
        )

    result["score"] = raw[hybrid_key]
    return result

train_scored = score_grants(train_df)
test_scored = score_grants(test_df)

# 5. Choose the classification threshold using TRAINING data only.
y_train = train_scored["Actual_Relevant"].astype(int).to_numpy()
train_scores = train_scored["score"].to_numpy()

# Include thresholds that classify all records as positive or negative.
candidates = np.unique(
    np.concatenate(([train_scores.min() - 1e-12],
                    train_scores,
                    [train_scores.max() + 1e-12]))
)

best_threshold = None
best_f1 = -1.0

for threshold in candidates:
    pred = (train_scores >= threshold).astype(int)
    score = f1_score(y_train, pred, zero_division=0)
    if score > best_f1:
        best_f1 = score
        best_threshold = float(threshold)

# 6. Apply the locked threshold to TEST data
test_scored["Predicted_Relevant"] = (
    test_scored["score"] >= best_threshold
).astype(int)

y_test = test_scored["Actual_Relevant"].astype(int).to_numpy()
y_pred = test_scored["Predicted_Relevant"].astype(int).to_numpy()
test_scores = test_scored["score"].to_numpy()

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, test_scores) if len(np.unique(y_test)) == 2 else np.nan

tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()

metrics = pd.DataFrame({
    "Metric": [
        "Selected threshold (training only)",
        "Training F1 at selected threshold",
        "Test accuracy",
        "Test precision",
        "Test recall",
        "Test F1-score",
        "Test ROC-AUC",
        "True negatives",
        "False positives",
        "False negatives",
        "True positives"
    ],
    "Value": [
        best_threshold, best_f1, accuracy, precision, recall, f1,
        roc_auc, tn, fp, fn, tp
    ]
})

confusion = pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=["Actual 0 - Not relevant", "Actual 1 - Relevant"],
    columns=["Predicted 0 - Not relevant", "Predicted 1 - Relevant"]
)

report = pd.DataFrame(
    classification_report(
        y_test, y_pred,
        labels=[0, 1],
        target_names=["Not relevant", "Relevant"],
        output_dict=True,
        zero_division=0
    )
).transpose().reset_index(names="Class")

# 7. Rank all grants by hybrid score and export the results
all_scored = pd.concat([train_scored, test_scored], ignore_index=True)
all_scored = all_scored.sort_values("score", ascending=False).reset_index(drop=True)
all_scored.insert(0, "Rank", np.arange(1, len(all_scored) + 1))

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    all_scored.to_excel(writer, sheet_name="All_Ranked_Grants", index=False)
    train_scored.to_excel(writer, sheet_name="Training_Scored", index=False)
    test_scored.to_excel(writer, sheet_name="Testing_Evaluation", index=False)
    metrics.to_excel(writer, sheet_name="Evaluation_Metrics", index=False)
    confusion.to_excel(writer, sheet_name="Confusion_Matrix")
    report.to_excel(writer, sheet_name="Classification_Report", index=False)

print("Evaluation complete.")
print(f"Training records: {len(train_scored)}")
print(f"Testing records: {len(test_scored)}")
print(f"Training-selected threshold: {best_threshold:.6f}")
print(f"Test accuracy:  {accuracy:.4f}")
print(f"Test precision: {precision:.4f}")
print(f"Test recall:    {recall:.4f}")
print(f"Test F1-score:  {f1:.4f}")
print(f"Test ROC-AUC:   {roc_auc:.4f}" if not np.isnan(roc_auc) else "Test ROC-AUC: N/A")
print(f"Saved results to: {output_file}")




Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Compute Scores: 100%|██████████| 4/4 [00:00<00:00, 38.02it/s]


Evaluation complete.
Training records: 60
Testing records: 40
Training-selected threshold: 0.363794
Test accuracy:  0.9250
Test precision: 0.9130
Test recall:    0.9545
Test F1-score:  0.9333
Test ROC-AUC:   0.9924
Saved results to: Grant_BGE_M3_Evaluation_Results.xlsx
